# 6차시 (상세) — 복잡한 모델이 늘 더 좋은 건 아니다

### 결정 트리 · 랜덤 포레스트 · 과적합 · 교차검증

> **상세 버전** — 기본 버전(`session6.ipynb`)에 트리·포레스트의 **작동 원리**를
> 파고드는 실습을 더했다. 지니 불순도 · 분할 탐색 · 가지치기 · 배깅 · OOB ·
> 변수 무작위 선택 · 분산 분해. 결론과 산출물은 기본 버전과 같다.

> **오늘 한 문장:** "5차시에 우리는 **직선**을 그었다. 오늘은 그 가정을 깨고
> **구부러진 모델**을 써 본다 — 그리고 그게 얼마나 도움이 되는지(혹은 안 되는지) 잰다."

5차시 마지막에 이렇게 물었다:

> "자아존중감이 **아주 낮은 구간에서만** 위험이 급격히 커지는 것이라면?"

오늘 확인한다 — **그런 구간이 실제로 데이터에 있다.** 그런데 그걸 잡아내는 모델이
반드시 이기는 것도 아니다. 오늘은 그 미묘한 사실을 **세 가지 증거**로 배운다.

오늘의 목표 4가지:

1. 로지스틱 회귀의 **숨은 가정(선형성)** 을 드러내고, 데이터가 그 가정을 따르는지 확인한다.
2. **결정 트리**가 어떻게 작동하는지 읽고, 트리가 발견한 **상호작용**을 해석한다.
3. **과적합**을 직접 만든다 — train AUC 1.0, CV AUC 0.52. ← 고비 1
4. 4개 모델을 정면 비교하고 **"복잡한 모델이 늘 더 좋은 건 아니다"** 를 판정한다. ← 고비 2

> 🔒 **오늘도 test 는 열지 않는다.** 오늘은 모델 **선택**이 끝나는 날이다.
> 바로 그래서 안 연다 — 훔쳐보고 싶은 유혹이 가장 큰 날이기 때문이다.

## 🗺️ 오늘의 위치 — 6차시

| 차시 | 심리학 | IT / ML |
|---|---|---|
| 1~3 ✅ | 척도 · 역채점 · 분포 · 상관 · α | pandas · join · 시각화 |
| 4 ✅ | 조작적 정의 · 임상 cut-off 와의 차이 | split · 불균형 · **데이터 누출** |
| 5 ✅ | 예측변수와 결과의 관계·방향성 | 로지스틱 · 계수 · 표준화 · 부트스트랩 |
| **6 (오늘)** | **심리 특성은 선형적으로 작동하는가** | **Decision Tree · Random Forest · 과적합 · CV** |
| 7 | 위험요인·보호요인 · 인과 vs 예측 | Permutation Importance · 오류 분석 |
| 8 | 결론 · 한계 · 윤리 서술 | 재현성 · **test 최종 1회** |

**오늘의 재료** — 4·5차시와 완전히 같은 데이터·라벨·분할이다.

- `modeling_frame.parquet` · `high_stress` 라벨(cutoff 1.500) · train 1,056 / test 265(봉인)
- `configs/modeling.yaml` 의 **하이퍼파라미터 그리드** — 숫자를 코드에 쓰지 않는다
- `maps_risk.models.build_models()` — Dummy / Logistic / Tree / Forest 넷

> 🔴 오늘의 규칙: **"성능 표에서 1등을 찾지 말고, 1등과 2등의 차이가 무엇을 사게 하는지 물어라."**

## Step 0 — 재료 확인

In [ ]:
!pip install pandas scikit-learn pyarrow matplotlib pyyaml -q
# Colab 에서 그림의 한글이 □ 로 깨지면 아래 한 줄을 실행하고 런타임을 재시작한다.
# !apt-get install -y fonts-nanum > /dev/null && rm -rf ~/.cache/matplotlib

In [ ]:
# ── 프로젝트 환경 자동 설정 (Colab / 로컬 공용) ───────────────────────
# 이 셀은 모든 차시 노트북 맨 위에 동일하게 들어간다. 그냥 실행만 하면 된다.
#
# Colab 사용법: 구글 드라이브('내 드라이브' 하위 2단계까지) 아무 곳에나
#   program5 zip 을 하나 올려 두면 된다. 이 셀이 드라이브를 mount 하고
#   /content 에 압축까지 풀어 준다. 런타임이 끊겨도 이 셀만 다시 실행하면
#   되고, 32MB zip 을 매번 재업로드할 필요가 없다.
import os, sys, glob, zipfile

DRIVE_ZIP_PATTERNS = [
    "/content/drive/MyDrive/program5*.zip",
    "/content/drive/MyDrive/*/program5*.zip",
    "/content/drive/MyDrive/*/*/program5*.zip",
]


# program5 로 인정하려면 이 4개가 다 있어야 한다.
# 왜 4개나 보나: 이름만 program5 인 '반쪽 폴더'(예전에 일부만 풀렸거나 업로드가 끊긴 것)를
#   붙잡으면 한참 뒤 4차시에서 "scripts/build_dataset.py 없음"으로 터진다. 여기서 거른다.
REQUIRED = ["AGENTS.md", "configs/variables.yaml", "configs/modeling.yaml",
            "scripts/build_dataset.py", "src/maps_risk/__init__.py"]


def missing_parts(path):
    """그 폴더에서 REQUIRED 중 빠진 파일 목록. 비어 있으면 온전한 프로젝트다."""
    return [f for f in REQUIRED if not os.path.isfile(os.path.join(path, *f.split("/")))]


def is_project(path):
    return not missing_parts(path)


def find_project():
    """이미 풀려 있는 program5 폴더를 후보 경로에서 찾는다.

    온전한 폴더만 고른다. program5 처럼 보이는데 반쪽인 폴더는 건너뛰되,
    **무엇이 없어서 건너뛰었는지 반드시 출력한다** — 조용히 넘어가면 원인 못 찾는다.
    """
    found, half, seen = None, [], set()
    for c in [".", "program5", "..", "../program5", "/content/program5",
              "/content/edu/program5", os.path.expanduser("~/program5")]:
        if not os.path.isdir(c):
            continue
        real = os.path.realpath(c)          # 같은 폴더를 두 경로로 가리키면 한 번만 본다
        if real in seen:
            continue
        seen.add(real)
        miss = missing_parts(c)
        if not miss:
            found = found or os.path.abspath(c)
        elif os.path.isfile(os.path.join(c, "AGENTS.md")):   # program5 인 척하는 반쪽 폴더
            half.append((os.path.abspath(c), miss))
    for path, miss in half:
        print("⚠️  반쪽 폴더라 건너뛴다:", path)
        print("     없는 것:", ", ".join(miss))
    return found


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def member_name(info):
    """UTF-8 플래그가 없는 zip 의 한글 파일명을 되살린다.

    맥 /usr/bin/zip 은 EFS(0x800) 플래그를 세우지 않는다. 그러면 zipfile 이
    이름을 cp437 로 잘못 디코딩해 'φòÖδ╢Ç…' 같은 폴더가 생기고, 뒤이어
    data/raw 스캔이 0개를 돌려준다. 원래 바이트로 되돌려 다시 읽는다.
    """
    if info.flag_bits & 0x800:          # 이미 UTF-8 로 제대로 읽힌 이름
        return info.filename
    try:
        raw = info.filename.encode("cp437")
    except UnicodeEncodeError:
        return info.filename
    for enc in ("utf-8", "cp949"):      # 한글 zip 은 이 둘 중 하나다
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            pass
    return info.filename


def setup_from_drive():
    """드라이브를 mount 하고 program5 zip 을 /content 에 푼다. 경로 또는 None."""
    from google.colab import drive
    drive.mount("/content/drive")   # 이미 붙어 있으면 그대로 통과한다

    # MyDrive 전체를 재귀 탐색하면 느리다 — 하위 2단계까지만 훑는다.
    zips = [z for p in DRIVE_ZIP_PATTERNS for z in sorted(glob.glob(p))]
    if not zips:
        print("⚠️  드라이브에서 program5*.zip 을 찾지 못했다.")
        print("   '내 드라이브' 또는 그 하위 2단계 폴더에 zip 을 두고 이 셀을 다시 실행하라.")
        print("   탐색한 위치:")
        for p in DRIVE_ZIP_PATTERNS:
            print("     ", p)
        return None

    src = zips[0]
    if len(zips) > 1:
        print(f"ℹ️  zip 후보 {len(zips)}개 중 첫 번째를 쓴다:",
              [os.path.basename(z) for z in zips])
    print(f"📦 {os.path.basename(src)} ({os.path.getsize(src) / 1e6:.1f} MB) → /content 에 푸는 중…")
    with zipfile.ZipFile(src) as zf:
        for info in zf.infolist():      # zip 안에 program5/ 폴더가 들어 있다
            info.filename = member_name(info)
            zf.extract(info, "/content")
    return find_project()


PROJECT = find_project()
if PROJECT is None and in_colab():
    PROJECT = setup_from_drive()

if PROJECT is None:
    print("🛑 프로젝트 폴더를 찾지 못했다. 아래 중 하나로 해결한다:")
    print("  (A) Colab: 구글 드라이브 '내 드라이브'에 program5 zip 을 올리고 이 셀 재실행")
    print("  (B) Colab: 좌측 파일창에 zip 을 올린 뒤  !unzip -q -o program5*.zip")
    print("      ※ 위에 '반쪽 폴더' 경고가 떴다면 그 폴더를 지우고 다시 풀어야 한다:")
    print("        !rm -rf /content/program5   ← 그 뒤 이 셀 재실행")
    print("  (C) 로컬 : program5 폴더 안(또는 그 상위)에서 노트북을 열었는지 확인")
    print("\n지금 /content 에 있는 것:", sorted(os.listdir("/content"))[:20]
          if os.path.isdir("/content") else "(없음)")
    # 여기서 멈춘다. 경고만 찍고 넘어가면 cwd 도 sys.path 도 안 잡힌 채로
    # 아래 셀들이 ModuleNotFoundError → FileNotFoundError 로 줄줄이 터진다.
    # sys.path 를 손으로 채워 봐야 cwd 가 여전히 /content 라 configs/*.yaml 을 못 읽는다.
    raise RuntimeError("program5 프로젝트 폴더를 찾지 못했다 — 위 안내대로 조치한 뒤 이 셀을 다시 실행하라.")

os.chdir(PROJECT)
src_dir = os.path.join(PROJECT, "src")
if src_dir not in sys.path:      # 셀을 여러 번 돌려도 중복 추가되지 않게
    sys.path.insert(0, src_dir)
print("✅ 프로젝트 경로:", PROJECT)


In [ ]:
# ── 차시 간 산출물 전달: 구글 드라이브에 저장/복원 ─────────────────────
# Colab 런타임은 끊기면 /content 가 사라진다. 그래서 "다음 차시가 필요로 하는 파일"은
# 내 드라이브에 따로 보관한다 — 그러면 차시 사이에 파일을 손으로 들고 다니지 않아도 된다.
#
#   저장 위치: 내 드라이브/program5_state/   (프로젝트와 같은 경로 구조로 쌓인다)
#     program5_state/configs/variables.yaml
#     program5_state/data/processed/modeling_frame.parquet
#     program5_state/reports/...
#
# 🔴 이 폴더에는 MAPS 원자료에서 파생된 파일이 들어간다. **개인 계정 안에만** 두고
#    링크 공유·양도하지 않는다 (MAPS 이용 조건). 공용 드라이브에 두지 말 것.
import filecmp as _filecmp
import glob as _glob
import os as _os
import shutil as _shutil

STATE_DIR = _os.environ.get("PROGRAM5_STATE_DIR")     # 로컬 테스트용 수동 지정


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _state_dir(create=False):
    """전달 폴더 경로. Colab 이 아니고 지정도 없으면 None 이라 그냥 건너뛴다."""
    global STATE_DIR
    if STATE_DIR:
        if create:
            _os.makedirs(STATE_DIR, exist_ok=True)
        return STATE_DIR
    if not _in_colab():
        return None
    from google.colab import drive
    drive.mount("/content/drive")          # 이미 붙어 있으면 그대로 통과한다
    STATE_DIR = "/content/drive/MyDrive/program5_state"
    _os.makedirs(STATE_DIR, exist_ok=True)
    return STATE_DIR


def handoff_push(patterns, label="다음 차시로 넘길 것을 드라이브에 저장"):
    """지금 만든 산출물을 드라이브에 저장한다.

    받는 것: 프로젝트 기준 상대경로 목록 (glob 가능. 예: reports/figures/*.png)
    돌려주는 것: 실제로 저장한 경로 리스트
    왜: 다음 차시가 이 파일을 "없으면 못 여는 재료"로 쓰기 때문이다.
    """
    print("📤 " + label)
    root = _state_dir(create=True)
    if root is None:
        print("   로컬 환경 — 저장을 건너뛴다 (파일이 이미 디스크에 그대로 남는다).")
        return []
    saved = []
    for pat in patterns:
        hits = sorted(_glob.glob(pat))
        if not hits:
            print("   ⬜ " + pat + " — 아직 없다 (이번 차시에서 만들지 않았다면 정상)")
            continue
        for src in hits:
            if not _os.path.isfile(src):
                continue
            dst = _os.path.join(root, src)
            _os.makedirs(_os.path.dirname(dst), exist_ok=True)
            _shutil.copy2(src, dst)
            saved.append(src)
            print("   ✅ " + src + "  →  드라이브")
    print("   저장 위치: " + root)
    return saved


def _yaml_completeness(path):
    """variables.yaml 이 얼마나 채워져 있는지 (게이트 열림, 검증된 구성개념 수).

    왜 필요한가: 드라이브에 **예전의 빈 variables.yaml** 이 남아 있는 경우가 있다.
    그걸 zip 의 검증본 위에 덮어쓰면 build_dataset.py 가 Human Review Gate 에서 멈춘다
    ("codebook_verified 가 false / 문항이 비어 있다"). 파일이 새것인지는 알 수 없어도
    **어느 쪽이 더 채워져 있는지**는 알 수 있다 — 덜 채워진 쪽으로는 덮어쓰지 않는다.
    """
    try:
        import yaml as _yaml
        d = _yaml.safe_load(open(path, encoding="utf-8")) or {}
    except Exception:
        return (0, 0)
    gate = bool((d.get("meta") or {}).get("codebook_verified"))
    n = sum(1 for sec in ("predictors", "optional_predictors")
            for spec in (d.get(sec) or {}).values()
            if spec.get("status") == "verified" and spec.get("items"))
    n += len((d.get("target") or {}).get("items") or [])
    return (int(gate), n)


# 파일별 '퇴보 방지' 검사. 드라이브 사본 점수가 지금 것보다 낮으면 그냥 둔다.
DOWNGRADE_GUARD = {"configs/variables.yaml": _yaml_completeness}


def handoff_pull(patterns, overwrite=True, label="지난 차시 산출물을 드라이브에서 복원"):
    """이번 차시에 필요한 파일을 드라이브에서 가져온다.

    받는 것: 상대경로 목록 (glob 가능), overwrite — 이미 있는 파일도 덮어쓸지 (기본 True)
    돌려주는 것: 실제로 가져온 경로 리스트
    왜 기본이 덮어쓰기인가: zip 안에 **같은 이름의 출발점 파일**이 이미 들어 있다
      (configs/variables.yaml · reports/model_metrics_cv.csv …). '없을 때만' 가져오면
      zip 의 옛 파일이 항상 이겨서 **지난 차시가 고친 내용이 영영 전달되지 않는다.**
      드라이브에 있는 것은 정의상 '지난 차시가 끝내고 밀어 넣은 최신본'이므로 그쪽을 쓴다.
    """
    print("📥 " + label)
    root = _state_dir()
    if root is None:
        print("   로컬 환경 — 복원을 건너뛴다 (디스크의 파일을 그대로 쓴다).")
        return []
    got = []
    for pat in patterns:
        hits = sorted(_glob.glob(_os.path.join(root, pat)))
        if not hits:
            print("   ⬜ " + pat + " — 드라이브에도 없다")
            continue
        for src in hits:
            rel = _os.path.relpath(src, root)
            exists = _os.path.exists(rel)
            if exists and _filecmp.cmp(src, rel, shallow=False):
                print("   ↩︎ " + rel + " — 드라이브와 내용이 같다 (그대로 둔다)")
                continue
            if exists and not overwrite:
                print("   ⚠️ " + rel + " — 드라이브 쪽과 다른데 덮어쓰지 않았다 (overwrite=False)")
                continue
            score = DOWNGRADE_GUARD.get(rel.replace(_os.sep, "/"))
            if exists and score and score(src) < score(rel):
                print("   🛡 " + rel + " — 드라이브 사본이 **더 비어 있다**. 지금 것을 그대로 쓴다.")
                print("       드라이브: " + str(score(src)) + " · 지금: " + str(score(rel))
                      + "   (게이트 열림, 검증된 구성개념 수)")
                print("       드라이브에 옛 파일이 남아 있는 것이다 — 이번 차시 끝에서 새것으로 덮인다.")
                continue
            _os.makedirs(_os.path.dirname(rel) or ".", exist_ok=True)
            _shutil.copy2(src, rel)
            got.append(rel)
            print(("   🔄 " if exists else "   ✅ ") + rel + "  ←  드라이브"
                  + ("  (zip 의 옛 파일을 덮어썼다)" if exists else ""))
    return got


def handoff_require(paths, hint=""):
    """이번 차시의 "없으면 못 여는 재료"를 확인한다. 없으면 이유를 알려준다."""
    missing = [p for p in paths if not _glob.glob(p)]
    if missing:
        print("\n🛑 이번 차시에 꼭 필요한 파일이 없다:")
        for m in missing:
            print("   -", m)
        if hint:
            print("   → " + hint)
        print("   → 지난 차시 노트북을 열어 **맨 끝의 '드라이브에 저장' 셀**을 실행한 뒤 돌아오라.")
    else:
        print("\n✅ 이번 차시에 필요한 재료가 전부 있다.")
    return not missing


handoff_pull([
    "configs/variables.yaml",
    "data/processed/modeling_frame.parquet",
])

handoff_require([
    "configs/variables.yaml",
    "data/processed/modeling_frame.parquet",
],
    hint="지난 차시 노트북 맨 끝의 '드라이브에 저장' 셀을 실행하면 여기서 자동으로 복원된다")


In [ ]:
# 4·5차시와 똑같은 상태를 재현한다 (seed 고정 — 한 명도 다르지 않다)
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from maps_risk.config import load_configs
from maps_risk.dataset import make_high_stress_label, split_features

_, cfg = load_configs("configs")
frame = pd.read_parquet("data/processed/modeling_frame.parquet")
scores = frame["acculturative_stress_w6"]

idx_tr, idx_te = train_test_split(frame.index, test_size=cfg["test_size"],
                                  random_state=cfg["random_seed"],
                                  stratify=(scores >= scores.median()).astype(int))
frame["high_stress"], cutoff = make_high_stress_label(
    scores.loc[idx_tr], scores, cfg["target"]["high_stress_quantile"])

featsA, featsB = split_features(frame, "A"), split_features(frame, "B")
Xtr, ytr = frame.loc[idx_tr, featsA], frame.loc[idx_tr, "high_stress"]

# ★ 모든 모델이 '같은' 폴드로 평가되어야 비교가 성립한다 — cv 객체를 하나만 만들어 돌려쓴다
cv = StratifiedKFold(n_splits=cfg["cv"]["folds"], shuffle=True, random_state=cfg["random_seed"])

print(f"train {len(idx_tr)} · test {len(idx_te)}(봉인) · cutoff {cutoff:.3f} · 양성 {ytr.mean():.1%}")
print(f"Model A {len(featsA)}변수 · Model B {len(featsB)}변수 · {cfg['cv']['folds']}-fold CV (seed {cfg['random_seed']})")
print("\nmodeling.yaml 그리드:")
for k in ("decision_tree", "random_forest", "logistic_regression"):
    print(f"  {k:20s} {cfg['models'][k]}")

## Step 1 — 로지스틱의 숨은 가정: 마음은 직선인가

5차시에 우리가 쓴 로지스틱 회귀는 이렇게 생겼다:

```
로그오즈 = b0 + b1·자아존중감 + b2·친구지지 + …
```

이 식에는 **말하지 않은 가정**이 하나 있다.
**"자아존중감이 1점에서 2점으로 오를 때의 효과 = 3점에서 4점으로 오를 때의 효과"**
— 어느 구간에서든 **똑같은 만큼** 로그오즈가 변한다는 가정이다. 이것을 **선형성 가정**이라 한다.

심리학적으로 이게 그럴듯한가? 아마 아닐 것이다.
자아존중감이 바닥인 학생에게 1점의 차이는 클 것이고,
이미 높은 학생들 사이의 1점 차이는 별 의미가 없을 수 있다.

**확인하는 방법은 간단하다.** 변수를 5구간(5분위)으로 나눠, 각 구간의 **실제 고스트레스 비율**을
세어 본다. 비율이 계단처럼 **일정하게** 변하면 선형에 가깝고, 어느 구간에서 **뚝 떨어지면**
비선형이다.

In [ ]:
# ▶ 변수를 5분위로 나눠 구간별 실제 고스트레스 비율을 세어라
import matplotlib.pyplot as plt
from maps_risk import plots      # import 만 해도 한글 폰트가 잡힌다

show = ["self_esteem", "depression", "previous_acculturative_stress", "peer_support"]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.4))
for ax, v in zip(axes, show):
    col = frame.loc[idx_tr, v]
    q = pd.qcut(col, 5, labels=False, duplicates="drop")   # ← 5분위로 나눈다
    rate = ytr.groupby(q).mean()                            # 0/1 의 평균 = 그 구간의 고스트레스 비율
    ax.plot(rate.index, rate.values, "o-")
    ax.axhline(ytr.mean(), color="gray", ls="--", lw=.8)
    ax.set_title(v, fontsize=10); ax.set_xlabel("5분위 (낮음→높음)"); ax.set_ylim(0, .65)
    print(f"  {v:30s} " + " → ".join(f"{r:.3f}" for r in rate))
axes[0].set_ylabel("고스트레스 비율")
fig.tight_layout(); plt.show()

In [ ]:
# CHECK Step1
try:
    se = ytr.groupby(pd.qcut(frame.loc[idx_tr, "self_esteem"], 5, labels=False, duplicates="drop")).mean()
    dp = ytr.groupby(pd.qcut(frame.loc[idx_tr, "depression"], 5, labels=False, duplicates="drop")).mean()
    drop1 = se.iloc[0] - se.iloc[1]          # 1분위 → 2분위 낙폭
    drop_rest = se.iloc[1] - se.iloc[-1]     # 2분위 → 5분위 낙폭
    assert drop1 > drop_rest, "자아존중감은 첫 구간에서 가장 크게 떨어져야 한다"
    assert dp.is_monotonic_increasing, "우울은 단조 증가여야 한다"
    print(f"✅ PASS — 자아존중감: 1→2분위에서 {drop1:.3f} 떨어지고, 2→5분위 전체에서 {drop_rest:.3f}밖에 안 떨어진다.")
    print("   **역치 효과(threshold effect)** 다 — 낮은 쪽에서만 위험이 급등하고 위쪽은 평평하다.")
    print("   반면 우울은 단조 증가 — 직선 가정이 잘 맞는 변수도 있다. 변수마다 다르다.")
except Exception as e:
    print("❌ FAIL —", e, "\n힌트: 0/1 라벨의 '평균'이 곧 그 구간의 고스트레스 비율이다 → .mean()")

<details><summary>💡 해설 (펼쳐 보기)</summary>

```python
rate = ytr.groupby(q).mean()
```

0/1 라벨의 **평균**이 곧 그 구간의 고스트레스 **비율**이다.

실측:

| 변수 | 1분위 → 5분위 | 모양 |
|---|---|---|
| `self_esteem` | .503 → .316 → .248 → .260 → **.220** | **역치형** — 1분위에서만 급락, 그 뒤 평평 |
| `depression` | .226 → .241 → .369 → .430 → **.464** | **단조 증가** — 직선에 가깝다 |
| `previous_acculturative_stress` | .218 → .206 → .305 → .462 → **.558** | 단조, 뒤로 갈수록 **가팔라짐** |
| `peer_support` | .509 → .396 → .189 → .234 | **비단조** — 3분위에서 최저, 4분위에서 되오름 |

**5차시의 예고가 맞았다.** 자아존중감은 낮은 구간에서만 위험이 급등하고 그 위로는 거의 평평하다.
직선 하나로는 이 모양을 표현할 수 없다.

> 그런데 주의: **"비선형이 존재한다"와 "비선형 모델이 이긴다"는 다른 말이다.**
> 오늘 그 차이를 확인하게 된다.
</details>

## Step 2 — 결정 트리: 스무고개로 분류하기

**결정 트리(Decision Tree)** 는 직선을 긋는 대신 **질문을 던진다.**

```
친구지지가 4.07 이하인가?
 ├─ 예  →  자아존중감이 2.88 이하인가? → …
 └─ 아니오 → 우울이 1.25 이하인가? → …
```

스무고개와 똑같다. 각 질문마다 데이터가 둘로 갈리고, 마지막 칸(**잎, leaf**)에 도달하면
그 칸에 속한 학생들의 고스트레스 비율로 확률을 매긴다.

트리의 장점 두 가지:

1. **비선형을 자동으로 잡는다.** "자아존중감 2.88 이하"라는 **역치**를 스스로 찾는다.
2. **상호작용(interaction)** 을 잡는다 — "A인 학생들 사이에서는 B가 중요하고,
   A가 아닌 학생들 사이에서는 C가 중요하다"는 구조를 표현할 수 있다.
   로지스틱 회귀는 이런 걸 **직접 적어 주지 않으면** 못 잡는다.

In [ ]:
# 깊이 2짜리 아주 작은 트리를 만들어 구조를 읽어 본다 (Model B — 이전 스트레스 포함)
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.pipeline import Pipeline
from maps_risk.preprocessing import make_preprocessor

def build(clf, scale=False):
    """트리 계열은 표준화가 필요 없다 — 하지만 결측 대치는 필요하므로 Pipeline 은 그대로 쓴다."""
    return Pipeline([("prep", make_preprocessor(scale=scale)), ("clf", clf)])

tree_b = build(DecisionTreeClassifier(max_depth=2, class_weight="balanced",
                                      random_state=cfg["random_seed"])).fit(frame.loc[idx_tr, featsB], ytr)
print(export_text(tree_b.named_steps["clf"], feature_names=list(featsB), decimals=2))

### Step 2 해석 — 트리가 찾아낸 것

출력을 읽으면 이런 구조다:

```
이전 스트레스 ≤ 1.45 ?
 ├─ 예 (이전에 스트레스가 낮았던 학생들)
 │    └─ 자아존중감 ≤ 2.62 ?  →  예: 고스트레스 / 아니오: 일반
 └─ 아니오 (이전에 스트레스가 높았던 학생들)
      └─ 우울 ≤ 1.25 ?        →  예: 일반 / 아니오: 고스트레스
```

**첫 질문이 `previous_acculturative_stress` 다.** 5차시에서 계수 1위였던 그 변수를
트리도 독립적으로 첫 번째로 골랐다 — 서로 다른 방법이 같은 결론에 도달했다.

더 흥미로운 건 **두 번째 층**이다. 트리는 두 집단에게 **서로 다른 질문**을 던진다:

> **이전에 스트레스가 낮았던 학생들** 사이에서는 → **자아존중감**이 갈림길이고,
> **이전에 스트레스가 높았던 학생들** 사이에서는 → **우울**이 갈림길이다.

이것이 **상호작용**이다. 로지스틱 회귀는 "자아존중감 계수 하나, 우울 계수 하나"만 주기 때문에
**이 구조를 말해 주지 못한다.** 트리가 진짜로 기여하는 지점이 여기다.

> ⚠️ 단, 이 구조는 **train 데이터 하나**에서 나온 것이다. 트리는 데이터가 조금만 바뀌어도
> 분기가 통째로 달라지는 **불안정한** 모델이다 (5차시의 부트스트랩과 같은 문제).
> 그래서 이 그림은 **가설**로 읽고, 확정된 발견으로 읽지 않는다.

## Step 2 심화 ① — 트리는 질문을 **어떻게 고르는가**

방금 트리는 `previous_acculturative_stress <= 1.45` 를 첫 질문으로 골랐다.
**이 변수와 이 숫자는 누가 정했나?** 우리가 정해 준 적이 없다. 트리가 계산해서 골랐다.

기준은 하나다 — **"나누고 나면 각 칸이 더 순수해지는가."**
'순수하다' = 한 칸 안에 고스트레스만, 또는 일반만 모여 있다는 뜻이다.
섞인 정도를 재는 자가 **지니 불순도(Gini impurity)** 다.

```
지니 = 1 − (고스트레스 비율)² − (일반 비율)²

  반반 섞임 → 1 − .5² − .5² = 0.500   ← 가장 지저분하다
  한 쪽만   → 1 − 1²  − 0²  = 0.000   ← 완벽하게 순수하다
```

**정보 이득(information gain)** = 분할 전 지니 − 분할 후 지니(자식들의 **가중** 평균).
트리는 이 값이 가장 큰 분할을 고른다. 직접 계산해 보자.

In [ ]:
# ▶ 지니를 손으로 계산해 sklearn 이 고른 분할과 맞춰 본다
def gini(y):
    """0/1 라벨 한 뭉치의 지니 불순도."""
    p = y.mean()
    return 1 - p**2 - (1 - p)**2

# 깊이 1 트리 = 딱 한 번만 나눈다 → 첫 분할이 무엇인지 바로 보인다
stump = build(DecisionTreeClassifier(max_depth=1, class_weight="balanced",
                                     random_state=cfg["random_seed"])).fit(Xtr, ytr)
t = stump.named_steps["clf"].tree_
feat, thr = featsA[t.feature[0]], t.threshold[0]
print(f"트리가 고른 첫 분할: {feat} <= {thr:.4f}\n")

# 같은 분할을 우리 손으로 재현한다 (결측은 트리와 같은 방식으로 중앙값 대치)
Xi = pd.DataFrame(stump.named_steps["prep"].transform(Xtr), columns=featsA, index=Xtr.index)
left = Xi[feat] <= thr

g_root = gini(ytr)
g_left, g_right = gini(ytr[left.values]), gini(ytr[~left.values])
w_left = left.mean()
g_child = w_left * g_left + (1 - w_left) * g_right

print(f"  분할 전  n {len(ytr):,}  양성 {ytr.mean():.4f}  지니 {g_root:.4f}")
print(f"  왼쪽     n {left.sum():,}  양성 {ytr[left.values].mean():.4f}  지니 {g_left:.4f}")
print(f"  오른쪽   n {(~left).sum():,}  양성 {ytr[~left.values].mean():.4f}  지니 {g_right:.4f}")
print(f"\n  가중 평균 자식 지니 = {w_left:.3f}·{g_left:.4f} + {1-w_left:.3f}·{g_right:.4f} = {g_child:.4f}")
print(f"  정보 이득 = {g_root:.4f} − {g_child:.4f} = {g_root - g_child:.4f}")

In [ ]:
# CHECK Step2심화-1
try:
    assert abs(g_root - 0.4469) < 5e-4, f"루트 지니가 {g_root:.4f} (기대 .4469)"
    assert g_child < g_root, "분할 후 지니가 분할 전보다 낮아야 한다"
    print(f"✅ PASS — 분할로 지니가 {g_root:.4f} → {g_child:.4f} 로 {g_root-g_child:.4f} 줄었다.")
    print("   트리는 이 '줄어드는 양'이 가장 큰 분할을 골랐다.")
    print(f"\n   그런데 sklearn 트리가 기록한 루트 지니는 {t.impurity[0]:.4f} 다 — 우리 계산과 다르다.")
    print("   틀린 게 아니다. 다음 셀에서 이유를 본다.")
except Exception as e:
    print("❌ FAIL —", e, "\n힌트: gini(y) 는 1 - p² - (1-p)² 이고 p 는 0/1 라벨의 평균이다")

<details><summary>💡 해설 — 왜 sklearn 의 지니는 0.5 인가 (펼쳐 보기)</summary>

우리 손계산은 **0.4469**, sklearn 트리 내부는 **0.5000**. 둘 다 맞다.

4차시에서 **불균형**을 배웠다. 우리 데이터의 양성은 **33.7%** 뿐이라, 그냥 두면
트리가 "전부 일반"으로 몰리는 쪽이 유리해진다. 그래서 우리는 계속
`class_weight="balanced"` 를 써 왔다 — **소수 집단 한 명을 더 무겁게 세는** 옵션이다.

가중치를 주면 트리가 보는 세상에서는 양성과 음성이 **정확히 반반**이 된다.
그래서 루트 지니가 `1 − .5² − .5² = 0.5` 가 된다.

| | 손계산 (가중치 없음) | sklearn (`class_weight="balanced"`) |
|---|---|---|
| 루트 | .4469 | **.5000** |
| 왼쪽 자식 | .4928 | .4771 |
| 오른쪽 자식 | .3491 | .4629 |

**고른 분할은 똑같다.** 자를 바꿔도 "어디서 가장 많이 줄어드는가"의 답은 같았다.

> 이것이 오늘의 작은 교훈이다 — **숫자가 안 맞을 때 먼저 의심할 것은 계산 실수가
> 아니라 "서로 다른 것을 재고 있는가"** 이다.
</details>

## Step 2 심화 ② — 후보를 **전부** 시도한다

트리는 똑똑하게 찍지 않는다. **가능한 분할을 전부 계산해 보고** 가장 좋은 하나를 고른다.

변수 하나에서 시도하는 임계값은 그 변수의 **서로 다른 값 사이사이**다.
그걸 18개 변수 전부에 대해 한다.

In [ ]:
# ▶ 분할 후보가 실제로 몇 개인지 세어 본다
cand = {f: Xi[f].nunique() for f in featsA}
total = sum(cand.values())

print(f"변수 {len(featsA)}개 · 임계값 후보 총 {total}개")
print(f"  가장 많은 변수: {max(cand, key=cand.get)} ({max(cand.values())}개)")
print(f"  가장 적은 변수: {min(cand, key=cand.get)} ({min(cand.values())}개)")
print(f"  첫 분할로 뽑힌 {feat} 는 {cand[feat]}개\n")

# 그 후보들을 우리가 직접 훑어 이득이 가장 큰 지점을 찾아본다 (트리가 하는 일 그대로)
vals = np.sort(Xi[feat].unique())
mids = (vals[:-1] + vals[1:]) / 2          # 값과 값 사이의 중점이 후보다
best = max(((gini(ytr) - ((Xi[feat] <= m).mean() * gini(ytr[(Xi[feat] <= m).values])
                          + (Xi[feat] > m).mean() * gini(ytr[(Xi[feat] > m).values])), m)
            for m in mids), key=lambda p: p[0])
print(f"{feat} 안에서 이득이 가장 큰 임계값: {best[1]:.4f} (이득 {best[0]:.4f})")
print(f"트리가 고른 값                  : {thr:.4f}")
print("\n→ 트리는 이 전수 탐색을 모든 변수에 대해 하고, 그중 최고 하나만 채택한다.")

## Step 2 심화 ③ — 언제 멈추게 할 것인가 (가지치기)

Step 3 에서 `max_depth` 로 과적합을 막았다. 손잡이는 그것만이 아니다.

**`min_samples_leaf`** — "잎 하나에 최소 몇 명은 있어야 한다"는 규칙이다.
한 명짜리 잎을 못 만들게 하면 트리는 저절로 얕아진다.

In [ ]:
# ▶ 잎에 최소 인원을 요구하면 과적합이 얼마나 잡히나
from sklearn.model_selection import cross_val_score   # 여기서 처음 쓴다

rows = []
for msl in (1, 5, 20, 50, 100):
    est = build(DecisionTreeClassifier(min_samples_leaf=msl, class_weight="balanced",
                                       random_state=cfg["random_seed"]))
    auc = cross_val_score(est, Xtr, ytr, cv=cv, scoring="roc_auc").mean()
    leaves = est.fit(Xtr, ytr).named_steps["clf"].get_n_leaves()
    rows.append({"min_samples_leaf": msl, "CV_AUC": auc, "리프수": leaves})

prune_tbl = pd.DataFrame(rows)
print(prune_tbl.round(4).to_string(index=False))
print("\n제한이 없으면(1) 리프 299개 · CV .5185 — Step 3 의 '깊이 제한 없음'과 같은 모델이다.")
print("잎마다 100명을 요구하면 리프 8개로 줄고 CV 가 .6352 까지 회복된다.")

**단조롭지 않다는 점**을 짚고 넘어가자 — `min_samples_leaf=50` 에서 한 번 내려간다(.6003).

"많이 자를수록 좋다"가 아니다. **적당한 지점이 있고, 그 지점은 데이터마다 다르다.**
그래서 우리는 이 값을 눈으로 고르지 않고 **`modeling.yaml` 그리드에 넣어 CV 에게 고르게** 한다.
Step 5 의 `GridSearchCV` 가 하는 일이 정확히 그것이다.

## Step 2 심화 ④ — 트리의 약점: **흔들린다** ⚠️

Step 2 에서 "이 구조는 가설로 읽어라"라고 했다. **왜 그런지 실제로 재 본다.**

같은 데이터를 조금씩 다르게 뽑아(부트스트랩) 트리를 40번 다시 학습시키면,
**첫 질문**이 몇 가지로 갈릴까? 예측해 보고 실행하자. 🖐

In [ ]:
# ▶ 재표집 40회 — '첫 질문'은 얼마나 바뀌는가
rng = np.random.default_rng(cfg["random_seed"])
first_splits = []
for _ in range(40):
    b = rng.integers(0, len(Xi), len(Xi))          # 복원추출
    t_b = DecisionTreeClassifier(max_depth=2, class_weight="balanced",
                                 random_state=cfg["random_seed"]).fit(Xi.iloc[b], ytr.iloc[b])
    first_splits.append(featsA[t_b.tree_.feature[0]])

vc = pd.Series(first_splits).value_counts()
print("40번 재표집했을 때 '첫 질문'으로 뽑힌 변수:\n")
print(vc.to_string())
print(f"\n→ {vc.size}가지로 갈린다. 데이터가 조금만 달라져도 트리의 얼굴이 바뀐다.")

In [ ]:
# CHECK Step2심화-2
try:
    assert vc.size >= 3, f"첫 분기 변수가 {vc.size}가지 — 최소 3가지는 나와야 한다"
    top1 = vc.index[0]
    share = vc.iloc[0] / vc.sum()
    print(f"✅ PASS — 첫 질문이 {vc.size}가지로 갈린다. 1위 {top1} 도 {share:.0%} 에 그친다.")
    print("   '이 변수가 가장 중요하다'를 트리 그림 하나로 주장하면 안 되는 이유다.")
    print("   → 그래서 5차시 계수와 '겹치는지'를 봤고, 7차시에 다시 한 번 교차 확인한다.")
except Exception as e:
    print("❌ FAIL —", e)

<details><summary>💡 해설 — 트리의 약점은 사실 하나로 모인다 (펼쳐 보기)</summary>

| 약점 | 정체 |
|---|---|
| 데이터가 바뀌면 구조가 통째로 바뀐다 | **분산이 크다** |
| 깊이를 풀면 바로 과적합한다 | **분산이 크다** |
| 확률 추정이 거칠다 (잎 단위 비율) | **분산이 크다** |

트리는 **편향이 낮고 분산이 높은** 모델이다.
"어떤 모양이든 그릴 수 있지만(저편향), 데이터를 조금만 바꾸면 다른 걸 그린다(고분산)."

통계에서 **분산이 큰 추정량을 다루는 표준 처방**은 하나다 — **여러 개를 평균 낸다.**

3차시 심리척도가 정확히 그 논리였다. 문항 하나(단일 측정)는 흔들리지만
여러 문항의 평균은 오차가 상쇄돼 안정된다.

**랜덤 포레스트는 트리를 개선한 모델이 아니다.** 트리를 **그대로 두고**
여러 그루를 평균 내는 장치다. Step 4 에서 그 장치를 뜯어본다.
</details>

## Step 3 — 과적합: 트리를 자라게 두면 ⚠️ (첫 봉우리)

트리는 질문을 계속 던질 수 있다. 깊이 제한을 풀면 **모든 학생을 하나씩 다른 칸에** 넣을 때까지
자란다. 그러면 train 데이터는 **100% 맞힌다.**

4차시에서 이 숫자를 본 적이 있다 — **AUC 1.0**. 그때는 **누출** 때문이었다.
오늘은 다른 병으로 같은 숫자가 나온다: **과적합(overfitting)**.

> **과적합** = 훈련 데이터의 **우연한 무늬까지 외워 버려서**, 새 데이터에서는 오히려 못 맞히는 것.
> 4차시의 기출문제 비유를 다시 쓰면 — **기출문제 100개를 통째로 암기한 학생**이다.

깊이를 1부터 끝까지 늘려 가며, **train 점수와 CV 점수를 나란히** 본다.

In [ ]:
# ▶ 깊이별로 train 점수와 CV 점수를 함께 재라
from sklearn.model_selection import cross_validate

rows = []
for d in (1, 2, 3, 4, 5, 8, 12, None):
    est = build(DecisionTreeClassifier(max_depth=d, class_weight="balanced",
                                       random_state=cfg["random_seed"]))
    r = cross_validate(est, Xtr, ytr, cv=cv, scoring="roc_auc",
                       return_train_score=True)           # train 점수도 함께 받는다
    n_leaves = est.fit(Xtr, ytr).named_steps["clf"].get_n_leaves()
    rows.append({"max_depth": str(d), "train_AUC": r["train_score"].mean(),
                 "CV_AUC": r["test_score"].mean(),
                 "차이": r["train_score"].mean() - r["test_score"].mean(), "리프수": n_leaves})

depth_tbl = pd.DataFrame(rows)
print(depth_tbl.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(range(len(depth_tbl)), depth_tbl["train_AUC"], "o-", label="train (외운 것)")
ax.plot(range(len(depth_tbl)), depth_tbl["CV_AUC"], "s-", label="CV (새 데이터)")
ax.set_xticks(range(len(depth_tbl)), depth_tbl["max_depth"])
ax.set_xlabel("트리 깊이"); ax.set_ylabel("AUC"); ax.legend(); ax.set_title("깊어질수록 벌어진다 — 과적합")
fig.tight_layout(); plt.show()

In [ ]:
# CHECK Step3
try:
    best_row = depth_tbl.loc[depth_tbl["CV_AUC"].idxmax()]
    deep = depth_tbl[depth_tbl["max_depth"] == "None"].iloc[0]
    assert deep["train_AUC"] > 0.99, "제한 없는 트리는 train 을 거의 완벽히 맞혀야 한다"
    assert deep["CV_AUC"] < 0.56, "그런데 CV 는 동전 던지기 수준으로 떨어져야 한다"
    assert best_row["max_depth"] in ("2", "3"), f"CV 최고는 얕은 트리여야 한다 (지금 {best_row['max_depth']})"
    print(f"✅ PASS — 깊이 제한 없음: train AUC {deep['train_AUC']:.4f} · CV AUC {deep['CV_AUC']:.4f} "
          f"(리프 {int(deep['리프수'])}개)")
    print(f"   CV 최고는 깊이 {best_row['max_depth']} 에서 {best_row['CV_AUC']:.4f} — **가장 단순한 축**이다.")
    print("   → train 점수는 깊어질수록 계속 오른다. 그런데 CV 는 어느 지점부터 **내려간다.**")
    print("   4차시의 AUC 1.0 은 누출이었고, 오늘의 1.0 은 과적합이다 — 병은 다르지만 증상은 같다.")
except Exception as e:
    print("❌ FAIL —", e, "\n힌트: cross_validate(..., return_train_score=True)")

<details><summary>💡 해설 (펼쳐 보기)</summary>

```python
return_train_score=True
```

실측:

| 깊이 | train AUC | CV AUC | 차이 | 리프 수 |
|---|---|---|---|---|
| 1 | .6199 | .6067 | .013 | 2 |
| **2** | .6637 | **.6355** | .028 | 4 |
| 3 | .6937 | .6151 | .079 | 8 |
| 5 | .7816 | .5892 | .192 | 30 |
| 8 | .9127 | .5312 | .382 | 96 |
| **없음** | **1.0000** | **.5185** | **.482** | **299** |

**train 점수는 끝까지 오르고, CV 점수는 깊이 2에서 꺾인다.** 이 그림이 과적합의 표준 형태다.

깊이 제한이 없으면 리프가 **299개** — train 1,056명을 299칸에 나눠 담았다.
train AUC 는 **1.0000** 인데 CV 는 **.5185**, 거의 **동전 던지기**다.

> 🔴 4차시에서 배운 문장이 여기서 다시 쓰인다: **"성능이 좋아 보이면 의심하라."**
> 그때는 누출이었고 지금은 과적합이다. **train 점수만 보면 둘 다 못 알아챈다.**
> 그래서 우리는 항상 **CV 점수를 옆에 둔다.**
</details>

## Step 4 — 랜덤 포레스트: 나무 한 그루 대신 숲

트리 하나는 불안정하다 — 데이터가 조금만 바뀌어도 분기가 통째로 달라지고, 깊어지면 외워 버린다.

**랜덤 포레스트(Random Forest)** 의 아이디어는 단순하다:

```
① 데이터를 조금씩 다르게 뽑아 (부트스트랩 — 5차시에 배운 그것!)
② 변수도 일부만 무작위로 골라 주고
③ 트리를 300그루 키운 뒤
④ 300개의 답을 평균 낸다
```

**왜 평균이 나은가.** 3차시의 심리척도 논리와 정확히 같다 —
"한 번의 측정은 흔들리지만, 여러 문항의 평균은 오차가 상쇄돼 안정된다."
트리 한 그루의 우연한 실수들이 300그루를 평균 내면 서로 상쇄된다.

깊이를 늘려 가며 포레스트도 과적합하는지 본다.

In [ ]:
# 포레스트도 깊이를 늘리면 과적합할까?
from sklearn.ensemble import RandomForestClassifier

rows = []
for d in (3, 5, 8, None):
    est = build(RandomForestClassifier(n_estimators=cfg["models"]["random_forest"]["n_estimators"],
                                       max_depth=d, class_weight="balanced",
                                       random_state=cfg["random_seed"], n_jobs=-1))
    r = cross_validate(est, Xtr, ytr, cv=cv, scoring="roc_auc", return_train_score=True)
    rows.append({"max_depth": str(d), "train_AUC": r["train_score"].mean(),
                 "CV_AUC": r["test_score"].mean(),
                 "차이": r["train_score"].mean() - r["test_score"].mean()})
forest_tbl = pd.DataFrame(rows)
print(forest_tbl.round(4).to_string(index=False))

print("\n비교 — 깊이 제한이 없을 때 CV AUC:")
print(f"  단일 트리   : {depth_tbl[depth_tbl['max_depth']=='None']['CV_AUC'].iloc[0]:.4f}   ← 동전 던지기 수준")
print(f"  랜덤 포레스트: {forest_tbl[forest_tbl['max_depth']=='None']['CV_AUC'].iloc[0]:.4f}   ← 훨씬 덜 무너진다")
print("\n→ 포레스트도 train 은 1.0 까지 외운다. 하지만 CV 가 덜 떨어진다 — 평균이 과적합을 완충한다.")

## Step 4 심화 ① — 배깅: 그루마다 **다른 표본**을 준다

300그루를 **같은 데이터**로 키우면 어떻게 될까? 트리는 결정적이다 —
**똑같은 트리 300개**가 나온다. 평균 내 봐야 한 그루와 같다.

그래서 그루마다 데이터를 흔든다. **부트스트랩(bootstrap)** — 1,056명에서
**복원추출**로 다시 1,056명을 뽑는다. 같은 학생이 두 번 뽑히기도 하고, 아예 안 뽑히기도 한다.

> 5차시에서 신뢰구간을 만들 때 쓴 그 기법이다. 같은 도구가 여기서는 **모델을 다양하게
> 만드는** 데 쓰인다.

**질문**: 복원추출로 n명을 뽑으면, 원래 n명 중 몇 %가 표본에 들어갈까? 🖐

In [ ]:
# ▶ 부트스트랩 표본에 실제로 몇 %가 들어가나
rng = np.random.default_rng(cfg["random_seed"])
n = len(Xtr)
uniq = [len(np.unique(rng.integers(0, n, n))) / n for _ in range(200)]

print(f"n = {n:,} · 복원추출 200회 반복")
print(f"  표본에 들어간 비율 (실측 평균) : {np.mean(uniq):.4f}")
print(f"  이론값 1 − 1/e                 : {1 - np.exp(-1):.4f}")
print(f"  한 번도 안 뽑힌 비율 (OOB)     : {1 - np.mean(uniq):.4f}")
print("\n→ 그루마다 약 37% 의 학생은 '처음 보는 데이터'로 남는다.")
print("   이 남은 학생들을 out-of-bag(OOB) 이라고 부른다.")

## Step 4 심화 ② — 안 뽑힌 37% 가 **공짜 검증**이 된다

각 그루에게 OOB 학생들은 학습에 쓰이지 않은 데이터다.
그러니 **그 학생들로 채점하면 그것이 곧 검증 점수**다 —
CV 처럼 모델을 5번 다시 학습시킬 필요가 없다.

sklearn 은 `oob_score=True` 한 줄로 이걸 계산해 준다.

In [ ]:
# ▶ OOB 점수 vs 교차검증 점수
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score

rf_oob = RandomForestClassifier(n_estimators=300, max_depth=3, class_weight="balanced",
                                random_state=cfg["random_seed"], n_jobs=-1,
                                bootstrap=True, oob_score=True)
prep = make_preprocessor(scale=False)
rf_oob.fit(prep.fit_transform(Xtr), ytr)

oob_auc = roc_auc_score(ytr, rf_oob.oob_decision_function_[:, 1])
cv_auc = cross_val_score(build(RandomForestClassifier(
    n_estimators=300, max_depth=3, class_weight="balanced",
    random_state=cfg["random_seed"], n_jobs=-1)), Xtr, ytr, cv=cv, scoring="roc_auc").mean()

print(f"OOB AUC          {oob_auc:.4f}   (추가 학습 0회 — 이미 만든 숲에서 바로 나온다)")
print(f"CV AUC (5-fold)  {cv_auc:.4f}   (5번 다시 학습해서 잰 값)")
print(f"차이             {abs(oob_auc - cv_auc):.4f}")
print("\n→ OOB 는 CV 의 값싼 대용품으로 쓸 만하다. 탐색 단계에서 시간을 크게 아낀다.")

> ⚠️ **그런데 우리 비교표는 왜 계속 CV 인가?**
>
> OOB 는 **포레스트에만 있다.** 로지스틱에도 Dummy 에도 없다.
> 모델끼리 비교하려면 **같은 자로** 재야 한다 — 그래서 Step 5 의 표는 전부 CV 다.
> (Step 5 슬라이드의 `CV=CV` 자물쇠가 이 이야기였다.)

## Step 4 심화 ③ — 변수까지 **무작위로** 고른다

표본만 흔들면 부족하다. **강한 변수 하나**가 있으면 그루 대부분이 그걸 첫 분기로
잡아서, 결국 **서로 닮은 트리**가 된다. 닮은 것끼리 평균 내면 이득이 없다.

그래서 랜덤 포레스트는 **분기마다** 전체 18개 중 **일부만** 후보로 보여 준다(`max_features`).
강한 변수가 후보에 없는 분기에서는 **2순위 변수가 기회를 얻는다.**

효과를 직접 재 보자 — 트리들이 서로 얼마나 닮았는지를 **예측값의 상관 ρ** 로 잰다.

In [ ]:
# ▶ max_features 를 바꿔가며 CV 와 '트리 간 상관 ρ' 를 함께 잰다  (30초쯤 걸린다)
from sklearn.model_selection import cross_val_score

rows = []
for mf in [1, 2, "sqrt", 6, 12, None]:
    est = RandomForestClassifier(n_estimators=300, max_depth=3, max_features=mf,
                                 class_weight="balanced", random_state=cfg["random_seed"], n_jobs=-1)
    auc = cross_val_score(build(est), Xtr, ytr, cv=cv, scoring="roc_auc").mean()

    est.fit(Xi, ytr)                                   # 상관을 보려고 전체 train 으로 한 번 더 학습
    P = np.array([t_.predict_proba(Xi.values)[:, 1] for t_ in est.estimators_])   # 300 × n
    C = np.corrcoef(P)
    rho = C[np.triu_indices_from(C, 1)].mean()         # 그루쌍 평균 상관

    rows.append({"max_features": str(mf), "후보변수": est.estimators_[0].max_features_,
                 "트리간_상관_rho": rho, "CV_AUC": auc})

mf_tbl = pd.DataFrame(rows)
print(mf_tbl.round(4).to_string(index=False))
print("\n→ 후보 변수를 늘릴수록 ρ 가 오른다 = 그루들이 서로 닮아 간다.")
print("   변수를 전부(18개) 보게 하면 ρ 가 높고 CV 가 가장 낮다(.6604).")
print("   기본값 sqrt(=4개)가 가장 좋다 — 이래서 그게 기본값이다.")

## Step 4 심화 ④ — 왜 평균이 분산을 줄이는가 (그리고 왜 ρ 가 관건인가)

트리 B개를 평균했을 때의 분산은 이렇게 쪼개진다:

```
Var( 트리 B개의 평균 )  =  ρ·σ²  +  (1 − ρ)·σ² / B

    σ² = 트리 하나의 분산 · ρ = 트리끼리의 상관 · B = 그루 수
```

- **B 를 키우면** 오른쪽 항 `(1−ρ)σ²/B` 가 0 으로 간다 → 그래서 그루를 늘려도 나빠지지 않는다.
- **그런데 왼쪽 항 `ρ·σ²` 는 남는다** → 그루를 아무리 늘려도 여기서 멈춘다.
- **그래서 ρ 를 낮춰야 한다** → 부트스트랩(다른 표본) + 변수 무작위 선택(다른 후보).

> 3차시 심리척도와 같은 논리다. 문항을 늘리면 α 가 오르지만, **문항들이 다 똑같은 걸
> 묻고 있으면** 아무리 늘려도 한계가 있다. 서로 다른 것을 묻는 문항이라야 평균이 이득이다.

**두 장치가 모두 ρ 를 낮추려고 있는 것이다.**

In [ ]:
# ▶ 그루를 늘리면 과적합하나? (깊이와 달리 이건 '위험한 손잡이'가 아니다)
from sklearn.model_selection import cross_val_score

rows = []
for n_tree in (1, 5, 25, 100, 300, 1000):
    est = build(RandomForestClassifier(n_estimators=n_tree, max_depth=3, class_weight="balanced",
                                       random_state=cfg["random_seed"], n_jobs=-1))
    rows.append({"n_estimators": n_tree,
                 "CV_AUC": cross_val_score(est, Xtr, ytr, cv=cv, scoring="roc_auc").mean()})

n_tbl = pd.DataFrame(rows)
print(n_tbl.round(4).to_string(index=False))
print("\n→ 1그루 .6251 → 100그루 부근에서 사실상 평평해진다. 1,000그루도 300그루와 같다.")
print("   max_depth 와 달리 n_estimators 는 '너무 크면 과적합하는' 손잡이가 아니다 —")
print("   더 키우면 계산 시간만 는다. 그래서 300 정도에서 멈춘다.")

## Step 4 심화 ⑤ — 흔들림을 **실제로** 재 본다

Step 2 심화에서 트리의 첫 질문이 40번 중 5가지로 갈리는 걸 봤다.
포레스트는 그걸 얼마나 잡아 줄까?

이번엔 구조가 아니라 **예측확률**로 잰다 —
같은 학생의 예측확률이 재표집에 따라 얼마나 흔들리는지.

In [ ]:
# ▶ 재표집 40회 — 같은 학생의 예측확률이 얼마나 흔들리나  (40초쯤 걸린다)
rng = np.random.default_rng(cfg["random_seed"])
boots = [rng.integers(0, len(Xi), len(Xi)) for _ in range(40)]

def spread(make_model):
    """재표집마다 다시 학습해, 같은 학생들에 대한 예측확률의 표준편차를 낸다."""
    P = np.array([make_model().fit(Xi.iloc[b], ytr.iloc[b]).predict_proba(Xi.values)[:, 1]
                  for b in boots])
    sd = P.std(axis=0, ddof=1)
    return sd.mean(), sd.max()

tree_sd, tree_max = spread(lambda: DecisionTreeClassifier(
    max_depth=2, class_weight="balanced", random_state=cfg["random_seed"]))
rf_sd, rf_max = spread(lambda: RandomForestClassifier(
    n_estimators=300, max_depth=3, class_weight="balanced",
    random_state=cfg["random_seed"], n_jobs=-1))

print("학생 1명의 예측확률이 재표집에 따라 흔들리는 폭 (표준편차)\n")
print(f"  단일 트리 (깊이 2)      평균 ±{tree_sd:.4f}   가장 심한 학생 ±{tree_max:.4f}")
print(f"  랜덤 포레스트 (300그루)  평균 ±{rf_sd:.4f}   가장 심한 학생 ±{rf_max:.4f}")
print(f"\n→ 트리가 {tree_sd/rf_sd:.2f}배 더 흔들린다.")

In [ ]:
# CHECK Step4심화
try:
    assert rf_sd < tree_sd, "포레스트가 트리보다 덜 흔들려야 한다"
    ratio = tree_sd / rf_sd
    assert ratio > 2, f"차이가 {ratio:.2f}배 — 2배 이상 나야 한다"
    print(f"✅ PASS — 포레스트의 흔들림이 트리의 1/{ratio:.2f} 다.")
    print("\n   여기서 오늘의 미묘한 지점이 하나 더 생긴다:")
    print("   Step 5 에서 포레스트가 로지스틱을 이긴 폭은 AUC +0.0116 뿐이었다.")
    print("   하지만 '안정성'은 성능 표에 아예 나오지 않는 이득이다.")
    print("   → 성능 표는 모델의 전부를 보여주지 않는다. 무엇이 표에 없는지도 물어야 한다.")
except Exception as e:
    print("❌ FAIL —", e)

### Step 4 심화 정리 — 포레스트의 네 손잡이

| 장치 | 무엇을 흔드나 | 왜 | 우리 설정 |
|---|---|---|---|
| **부트스트랩** | 학생 표본 | 그루마다 다른 세상을 보게 | 복원추출 1,056명 (유효 63.1%) |
| **max_features** | 분기별 후보 변수 | 강한 변수 독점을 막아 ρ 를 낮춤 | `sqrt` → 18개 중 **4개** |
| **n_estimators** | — | 평균의 표본 수 (많을수록 안정) | **300**그루 (100 이후 평평) |
| **max_depth** | — | 그루 하나의 복잡도 | CV 가 고른 **3** |

**포레스트가 산 것**: 안정성(흔들림 1/2.81) + CV AUC +0.0116
**포레스트가 판 것**: 그림으로 읽을 수 있던 **트리 한 그루** — 300그루는 그릴 수 없다.

> 이 거래가 우리 목적에 맞는지가 **Step 5 의 질문**이다. 바로 이어서 답한다.

## Step 5 — 4개 모델 정면 비교 🔍 (두 번째 봉우리)

이제 오늘의 본론이다. 네 모델을 **완전히 같은 조건**에서 비교한다:

- 같은 train 데이터(1,056명), 같은 **5개 폴드**(같은 `cv` 객체를 돌려 쓴다)
- 하이퍼파라미터는 `modeling.yaml` 의 그리드에서 **CV 로 선택**한다
- **Dummy 를 반드시 옆에 둔다** (4차시의 규칙)

In [ ]:
# ▶ modeling.yaml 의 그리드로 네 모델을 튜닝하고 CV 로 비교하라
from sklearn.model_selection import GridSearchCV
from maps_risk.models import build_models

def compare(cols, label):
    X = frame.loc[idx_tr, cols]
    out = []
    for name, (est, grid) in build_models(cfg).items():
        if grid:
            gs = GridSearchCV(est, grid, scoring="roc_auc", cv=cv, n_jobs=-1).fit(X, ytr)  # 모든 모델에 같은 폴드!
            chosen, auc = gs.best_params_, gs.best_score_
            fitted = gs.best_estimator_
        else:
            chosen, auc, fitted = {}, cross_validate(est, X, ytr, cv=cv, scoring="roc_auc")["test_score"].mean(), est
        extra = cross_validate(fitted, X, ytr, cv=cv, return_train_score=True,
                               scoring=["roc_auc", "average_precision", "recall", "balanced_accuracy"])
        out.append({"model_set": label, "model": name, "best_params": str(chosen),
                    "cv_roc_auc": round(auc, 4),
                    "cv_average_precision": round(extra["test_average_precision"].mean(), 4),
                    "cv_recall": round(extra["test_recall"].mean(), 4),
                    "cv_balanced_accuracy": round(extra["test_balanced_accuracy"].mean(), 4),
                    "train_roc_auc": round(extra["train_roc_auc"].mean(), 4)})
    return pd.DataFrame(out)

metrics = pd.concat([compare(featsA, "A"), compare(featsB, "B")], ignore_index=True)
print(metrics.to_string(index=False))

In [ ]:
# CHECK Step5
try:
    A = metrics[metrics.model_set == "A"].set_index("model")["cv_roc_auc"]
    assert A["Dummy"] == 0.5, "Dummy 는 0.5 여야 한다"
    assert A["DecisionTree"] < A["LogisticRegression"], \
        "단일 트리가 로지스틱보다 나빠야 한다 (실측: .6355 < .6535)"
    assert A["RandomForest"] > A["LogisticRegression"], \
        "포레스트는 로지스틱보다 조금 나아야 한다 (실측: .6651 > .6535)"
    gap = A["RandomForest"] - A["LogisticRegression"]
    print("✅ PASS — Model A 결과:")
    print(f"   Dummy {A['Dummy']:.4f} · 로지스틱 {A['LogisticRegression']:.4f} · "
          f"트리 {A['DecisionTree']:.4f} · 포레스트 {A['RandomForest']:.4f}")
    print(f"   ① 단일 트리는 로지스틱보다 **나쁘다** — 더 유연한 모델인데 더 못한다.")
    print(f"   ② 포레스트는 로지스틱보다 낫다 — 그런데 차이가 **{gap:+.4f}** 다.")
    print("   이 두 사실을 어떻게 읽어야 하나? 다음 셀에서 차이가 진짜인지부터 확인한다.")
except Exception as e:
    print("❌ FAIL —", e, "\n힌트: GridSearchCV(..., cv=cv) — 모든 모델이 같은 폴드를 써야 비교가 성립한다.")

In [ ]:
# 포레스트의 +0.012 는 진짜인가, 폴드 운인가? — 폴드별로 뜯어본다
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

log_est = build(LogisticRegression(max_iter=2000, class_weight="balanced",
                                   random_state=cfg["random_seed"], C=0.1), scale=True)
rf_est = build(RandomForestClassifier(n_estimators=300, max_depth=3, class_weight="balanced",
                                      random_state=cfg["random_seed"], n_jobs=-1))
a = cross_val_score(log_est, Xtr, ytr, cv=cv, scoring="roc_auc")
b = cross_val_score(rf_est, Xtr, ytr, cv=cv, scoring="roc_auc")
print("폴드별 AUC")
print("  로지스틱 : " + " ".join(f"{v:.3f}" for v in a) + f"   평균 {a.mean():.4f}")
print("  포레스트 : " + " ".join(f"{v:.3f}" for v in b) + f"   평균 {b.mean():.4f}")
print("  차이     : " + " ".join(f"{v:+.3f}" for v in b - a) +
      f"   → 포레스트 승 {int((b > a).sum())}/{len(a)} 폴드")

print("\nCV 분할 seed 를 바꿔도 순위가 유지되나?")
for sd in (0, 1, 7, 42, 123, 2024, 777):
    c2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=sd)
    x1 = cross_val_score(log_est, Xtr, ytr, cv=c2, scoring="roc_auc").mean()
    x2 = cross_val_score(rf_est, Xtr, ytr, cv=c2, scoring="roc_auc").mean()
    print(f"  seed={sd:<5d} 로지스틱 {x1:.4f} · 포레스트 {x2:.4f} → {'포레스트' if x2 > x1 else '로지스틱'} ({x2-x1:+.4f})")

### Step 5 해석 — "복잡한 모델이 늘 더 좋은 건 아니다"의 세 가지 증거

실측을 정직하게 정리하면 이렇다.

**증거 ① 단일 트리는 로지스틱보다 나쁘다** (.6355 vs .6535)
트리는 로지스틱보다 **훨씬 유연한** 모델이다. 비선형도 상호작용도 잡을 수 있다.
그런데 **더 못한다.** 유연함은 그 자체로 성능이 아니다 — 유연한 만큼 **흔들리기** 때문이다.

**증거 ② 복잡도를 늘리면 오히려 나빠진다** (Step 3)
깊이 2 → 없음으로 가면 CV AUC 가 .6355 → .5185 로 **떨어진다.**
"모델을 더 강력하게" 가 성능을 보장하지 않는다.

**증거 ③ 포레스트는 이긴다 — 그런데 +0.012 다**
여기서 정직해야 한다. 포레스트는 **5개 폴드 전부에서** 로지스틱을 이겼고,
CV 분할 seed 를 7가지로 바꿔도 6번 이겼다. **이 차이는 폴드 운이 아니다.**

그렇다면 포레스트를 써야 하나? **비용을 같이 봐야 한다.**

| | 로지스틱 회귀 | 랜덤 포레스트 |
|---|---|---|
| CV AUC (Model A) | .6535 | **.6651** (+.012) |
| 변수별 **방향**(+/−) | ✅ 계수 부호로 안다 | ❌ 없다 |
| 변수별 **크기** | ✅ 표준화 계수 | △ 7차시 permutation 으로 일부 |
| **불확실성** | ✅ 부트스트랩 신뢰구간 | ❌ 사실상 불가 |
| 사람이 읽을 수 있나 | ✅ 18줄짜리 표 | ❌ 나무 300그루 |

> 🔴 **오늘의 질문: AUC 0.012 를 위해 5차시의 그 표를 통째로 포기할 것인가?**
>
> 우리 연구 질문은 **RQ2 — "어떤 심리사회적 변수가 상대적으로 중요한가"** 다.
> 목적이 **해석**이므로 우리는 **로지스틱을 주 모델로** 삼고,
> 포레스트는 **"비선형을 넣어도 크게 좋아지지 않았다"는 근거**로 함께 보고한다.
>
> 이것은 **정답이 아니라 선택**이다. 목적이 "최대한 잘 맞히기"였다면 반대로 골랐을 것이다.
> 중요한 건 **고르고, 이유를 적는 것**이다.

## Step 6 — Model A vs B: 변수 구성이 바뀌면 순위도 바뀐다

같은 표의 Model B 를 보면 재미있는 일이 벌어진다.

In [ ]:
pivot = metrics.pivot(index="model", columns="model_set", values="cv_roc_auc")
pivot["B−A"] = pivot["B"] - pivot["A"]
print(pivot.round(4).to_string())
print("\n주목: Model A 에서 로지스틱보다 나빴던 단일 트리가, Model B 에서는 거의 같아진다.")
print("      이전 스트레스라는 '강한 단일 변수'가 생기자 트리가 그것을 첫 분기로 잡아 잘 작동한다.")
print("      → 모델의 우열은 고정된 것이 아니라 **어떤 변수를 주느냐에 따라 달라진다.**")

## Step 7 — 산출물, 그리고 왜 지금 test 를 열지 않는가

오늘로 **모델 선택이 끝났다.** 하이퍼파라미터도 정했고 주 모델도 정했다.
교과서적으로는 **바로 지금이 test 를 열 시점**이다. 그런데 우리는 열지 않는다.

이유는 4차시에 배운 것 그대로다:

> test 를 한 번 보면, **그다음 결정이 그 숫자에 영향받는다.**
> 7차시에 변수 중요도를 보다가 "이 변수를 빼면 어떨까" 하는 생각이 들 수 있다.
> 그때 test 점수를 이미 알고 있으면, 그 판단은 **더 이상 깨끗하지 않다.**

그래서 순서를 이렇게 둔다: **6차시 선택 → 7차시 해석 → 8차시에 test 를 딱 한 번.**

> 🔴 **훔쳐보고 싶은 유혹이 가장 큰 날이 바로 오늘이다.** 그래서 오늘 안 여는 것이 규칙이다.

In [ ]:
# 오늘의 산출물 — CV 기준 성능표 (test 점수가 아니다)
import os
os.makedirs("reports", exist_ok=True)
metrics.to_csv("reports/model_metrics_cv.csv", index=False)
print("✅ reports/model_metrics_cv.csv  (모든 숫자는 train 안 5-fold CV 값이다)")

fig, ax = plt.subplots(figsize=(7, 3.6))
w = 0.35; order = ["Dummy", "LogisticRegression", "DecisionTree", "RandomForest"]
for i, ms in enumerate(("A", "B")):
    v = metrics[metrics.model_set == ms].set_index("model").loc[order, "cv_roc_auc"]
    ax.bar(np.arange(len(order)) + i*w, v.values, w, label=f"Model {ms}")
ax.axhline(0.5, color="gray", ls="--", lw=.8)
ax.set_xticks(np.arange(len(order)) + w/2, ["Dummy", "Logistic", "Tree", "Forest"])
ax.set_ylabel("CV ROC-AUC"); ax.set_ylim(0.45, 0.75); ax.legend()
ax.set_title("네 모델 · 두 변수 세트 (train 5-fold CV)")
fig.tight_layout(); fig.savefig("reports/figures/model_comparison_cv.png", dpi=150)
plt.show()
print("✅ reports/figures/model_comparison_cv.png")

print(f"\n🔒 test {len(idx_te)}명 — 오늘도 열지 않았다. 8차시에 딱 한 번 연다.")

### ⚠️ 오늘 결과를 볼 때의 정직한 단서 하나

`GridSearchCV` 의 `best_score_` 는 **여러 후보 중 가장 좋았던 값**이다.
후보가 많을수록 "운 좋게 잘 나온 값"이 뽑힐 가능성도 커진다 —
포레스트·트리는 후보가 4개, 로지스틱은 3개, Dummy 는 0개다.

즉 **복잡한 모델 쪽이 아주 약간 유리하게 채점됐다.** 엄밀하게 하려면
**중첩 교차검증(nested CV)** — 튜닝용 CV 안에 평가용 CV 를 한 겹 더 — 을 써야 한다.

이 수업에서는 거기까지 가지 않는다. 대신 **그런 편향이 있다는 사실을 기록**한다.
포레스트의 우위가 +0.012 로 작다는 점을 감안하면, 이 편향은 **결론을 뒤집을 수 있는 크기**다.
8차시 한계 절에 적는다.

## 💾 다음 차시를 위해 — 드라이브에 저장

오늘 만든 것 중 **다음 차시가 재료로 쓰는 파일**을 내 드라이브(`program5_state/`)에 넣어 둔다.
이렇게 해 두면 런타임이 끊겨도, 다른 컴퓨터에서 열어도 **다음 차시가 그냥 시작된다.**

> 🔴 파생 파일이 들어가는 폴더다 — **개인 계정 안에만** 두고 링크 공유·양도하지 않는다.

In [ ]:
# 6차시 산출물 — 모델 비교표를 7·8차시가 이어받는다
handoff_push([
    "reports/model_metrics_cv.csv",
    "reports/figures/*.png",
])


## 🎯 회고 (5분)

1. 깊이 제한 없는 트리는 train AUC 가 **1.0** 이었다. 4차시에도 AUC 1.0 을 봤다.
   **두 경우는 어떻게 다른가?** 그리고 **공통점**은 무엇인가?
2. 단일 트리는 로지스틱보다 **유연한데 더 나빴다.** 왜 그런가?
3. 포레스트가 로지스틱보다 **일관되게** 0.012 만큼 낫다. 그런데도 우리는 로지스틱을
   주 모델로 골랐다. **정당한 선택인가?** 어떤 연구 목적이었다면 반대로 골라야 했나?

3번이 오늘의 핵심 감각이다 — **성능 표의 1등이 곧 답이 아니다. 목적이 답을 정한다.**

## 📝 과제
- 깊이별 train/CV 표를 보고 **"과적합이 시작되는 지점"** 을 짚고, 그 근거를 2문장으로
- Model B 깊이 2 트리의 **상호작용**을 심리학적으로 해석 (단, "가설"이라는 단서를 달 것)
- 우리 연구에서 로지스틱을 주 모델로 고른 이유를 **비용-편익 표**로 정리

## ▶️ 다음 (7차시)
> "오늘 모델을 골랐다. 다음엔 그 모델에게 **'무엇을 보고 판단했니?'** 라고 묻는다 —
> **Permutation Importance**. 그리고 5차시의 계수 순위와 비교한다.
> 두 방법이 같은 답을 주면 든든하고, 다르면 그 이유를 설명해야 한다.
> 마지막으로 가장 어려운 질문이 온다 — **이 중요도를 '위험요인'이라고 불러도 되는가?**